# Initiation Phase

## Import & Configuration

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1]

RAW_DIR = (
    PROJECT_ROOT
    / "assets"
    / "data"
    / "raw"
    / "sba"
)

STAGING_DIR = (
    PROJECT_ROOT
    / "assets"
    / "data"
    / "staging"
    / "sba"
)

STAGING_DIR.mkdir(parents=True, exist_ok=True)

## CSV Inventory

In [2]:
csv_files = sorted(RAW_DIR.glob("*.csv"))

inventory = []

for file in csv_files:
    inventory.append({
        "file_name": file.name,
        "file_size_mb": round(
            file.stat().st_size / (1024 ** 2), 2
        ),
        "program": (
            "7(a)" if "7a" in file.name.lower()
            else "504" if "504" in file.name.lower()
            else "Unknown"
        )
    })

df_inventory = pd.DataFrame(inventory)

display(df_inventory)

,file_name,file_size_mb,program
0,FOIA_504_FY1991_FY2009_asof_260930.csv,51.20,504
1,FOIA_504_FY2010_Present_asof_260930.csv,57.51,504
2,FOIA_7a_FY1991_FY1999_asof_260930.csv,139.75,7(a)
3,FOIA_7a_FY2000_FY2009_asof_260930.csv,303.93,7(a)
4,FOIA_7a_FY2010_FY2019_asof_260930.csv,243.47,7(a)
5,FOIA_7a_FY2020_Present_asof_260930.csv,179.75,7(a)


## Inventory Summary

In [3]:
print("Total CSV:", len(df_inventory))

print("\nJumlah file per program:")
print(df_inventory["program"].value_counts())

print("\nTotal ukuran:")
print(round(df_inventory["file_size_mb"].sum(), 2), "MB")

Total CSV: 6

Jumlah file per program:
program
7(a)    4
504     2
Name: count, dtype: int64

Total ukuran:
975.61 MB


# Process

## Schema Inspection

In [4]:
schema_data = {}

for file in csv_files:
    df_header = pd.read_csv(
        file,
        nrows=0
    )

    schema_data[file.name] = list(df_header.columns)

    print(f"\n{file.name}")
    print("Total columns:", len(df_header.columns))
    print(df_header.columns.tolist())


FOIA_504_FY1991_FY2009_asof_260930.csv
Total columns: 39
['AsOfDate', 'Program', 'LocationID', 'BorrName', 'BorrStreet', 'BorrCity', 'BorrState', 'BorrZip', 'CDC_Name', 'CDC_Street', 'CDC_City', 'CDC_State', 'CDC_Zip', 'ThirdPartyLender_Name', 'ThirdPartyLender_City', 'ThirdPartyLender_State', 'ThirdPartyDollars', 'GrossApproval', 'ApprovalDate', 'ApprovalFY', 'FirstDisbursementDate', 'ProcessingMethod', 'TermInMonths', 'NaicsCode', 'NaicsDescription', 'FranchiseCode', 'FranchiseName', 'ProjectCounty', 'ProjectState', 'SBADistrictOffice', 'CongressionalDistrict', 'BusinessType', 'BusinessAge', 'LoanStatus', 'PaidInFullDate', 'ChargeOffDate', 'GrossChargeOffAmount', 'JobsSupported', 'CollateralInd']

FOIA_504_FY2010_Present_asof_260930.csv
Total columns: 39
['AsOfDate', 'Program', 'LocationID', 'BorrName', 'BorrStreet', 'BorrCity', 'BorrState', 'BorrZip', 'CDC_Name', 'CDC_Street', 'CDC_City', 'CDC_State', 'CDC_Zip', 'ThirdPartyLender_Name', 'ThirdPartyLender_City', 'ThirdPartyLender_St

In [5]:
schema_sets = {
    name: set(columns)
    for name, columns in schema_data.items()
}

reference_name = csv_files[0].name
reference_columns = schema_sets[reference_name]

schema_comparison = []

for name, columns in schema_sets.items():
    schema_comparison.append({
        "file_name": name,
        "total_columns": len(columns),
        "missing_from_reference": len(
            reference_columns - columns
        ),
        "additional_columns": len(
            columns - reference_columns
        ),
        "same_schema": columns == reference_columns
    })

df_schema_comparison = pd.DataFrame(schema_comparison)

display(df_schema_comparison)

,file_name,total_columns,missing_from_reference,additional_columns,same_schema
0,FOIA_504_FY1991_FY2009_asof_260930.csv,39,0,0,True
1,FOIA_504_FY2010_Present_asof_260930.csv,39,0,0,True
2,FOIA_7a_FY1991_FY1999_asof_260930.csv,42,9,12,False
3,FOIA_7a_FY2000_FY2009_asof_260930.csv,42,9,12,False
4,FOIA_7a_FY2010_FY2019_asof_260930.csv,42,9,12,False
5,FOIA_7a_FY2020_Present_asof_260930.csv,42,9,12,False


In [6]:
all_columns = set.union(*schema_sets.values())

column_presence = pd.DataFrame({
    "column_name": sorted(all_columns)
})

for file_name, columns in schema_sets.items():
    column_presence[file_name] = (
        column_presence["column_name"].isin(columns)
    )

display(column_presence)

,column_name,FOIA_504_FY1991_FY2009_asof_260930.csv,FOIA_504_FY2010_Present_asof_260930.csv,FOIA_7a_FY1991_FY1999_asof_260930.csv,FOIA_7a_FY2000_FY2009_asof_260930.csv,FOIA_7a_FY2010_FY2019_asof_260930.csv,FOIA_7a_FY2020_Present_asof_260930.csv
0,ApprovalDate,True,True,True,True,True,True
1,ApprovalFY,True,True,True,True,True,True
2,AsOfDate,True,True,True,True,True,True
3,BankCity,False,False,True,True,True,True
4,BankFDICNumber,False,False,True,True,True,True
5,BankNCUANumber,False,False,True,True,True,True
6,BankName,False,False,True,True,True,True
7,BankState,False,False,True,True,True,True
8,BankStreet,False,False,True,True,True,True
9,BankZip,False,False,True,True,True,True


In [7]:
dtype_results = []

for file in csv_files:

    df_sample = pd.read_csv(
        file,
        nrows=1000,
        low_memory=False
    )

    for column in df_sample.columns:
        dtype_results.append({
            "file_name": file.name,
            "column_name": column,
            "sample_dtype": str(df_sample[column].dtype),
            "sample_missing": df_sample[column].isna().sum()
        })

df_dtype = pd.DataFrame(dtype_results)

display(df_dtype)

,file_name,column_name,sample_dtype,sample_missing
0,FOIA_504_FY1991_FY2009_asof_260930.csv,AsOfDate,object,0
1,FOIA_504_FY1991_FY2009_asof_260930.csv,Program,int64,0
2,FOIA_504_FY1991_FY2009_asof_260930.csv,LocationID,int64,0
3,FOIA_504_FY1991_FY2009_asof_260930.csv,BorrName,object,0
4,FOIA_504_FY1991_FY2009_asof_260930.csv,BorrStreet,object,0
...,...,...,...,...
241,FOIA_7a_FY2020_Present_asof_260930.csv,GrossChargeOffAmount,float64,0
242,FOIA_7a_FY2020_Present_asof_260930.csv,RevolverStatus,object,0
243,FOIA_7a_FY2020_Present_asof_260930.csv,JobsSupported,float64,0
244,FOIA_7a_FY2020_Present_asof_260930.csv,CollateralInd,object,0


## Consistency Check

In [8]:
program_schemas = {
    "7a": [],
    "504": []
}

for file_name, columns in schema_data.items():

    if "7a_" in file_name.lower():
        program_schemas["7a"].append(
            (file_name, set(columns))
        )

    elif "504_" in file_name.lower():
        program_schemas["504"].append(
            (file_name, set(columns))
        )


for program, schemas in program_schemas.items():

    print(f"\n===== SBA {program.upper()} =====")

    reference_file, reference_set = schemas[0]

    for file_name, current_set in schemas:
        print(f"\nFile: {file_name}")
        print("Missing:", sorted(reference_set - current_set))
        print("Additional:", sorted(current_set - reference_set))
        print("Same schema:", reference_set == current_set)


===== SBA 7A =====

File: FOIA_7a_FY1991_FY1999_asof_260930.csv
Missing: []
Additional: []
Same schema: True

File: FOIA_7a_FY2000_FY2009_asof_260930.csv
Missing: []
Additional: []
Same schema: True

File: FOIA_7a_FY2010_FY2019_asof_260930.csv
Missing: []
Additional: []
Same schema: True

File: FOIA_7a_FY2020_Present_asof_260930.csv
Missing: []
Additional: []
Same schema: True

===== SBA 504 =====

File: FOIA_504_FY1991_FY2009_asof_260930.csv
Missing: []
Additional: []
Same schema: True

File: FOIA_504_FY2010_Present_asof_260930.csv
Missing: []
Additional: []
Same schema: True


In [9]:
schema_7a = program_schemas["7a"][0][1]
schema_504 = program_schemas["504"][0][1]

print("Columns only in SBA 7(a):")
print(sorted(schema_7a - schema_504))

print("\nColumns only in SBA 504:")
print(sorted(schema_504 - schema_7a))

print("\nCommon columns:")
print(sorted(schema_7a & schema_504))

Columns only in SBA 7(a):
['BankCity', 'BankFDICNumber', 'BankNCUANumber', 'BankName', 'BankState', 'BankStreet', 'BankZip', 'FixedorVariableInterestInd', 'InitialInterestRate', 'RevolverStatus', 'SBAGuaranteedApproval', 'SoldSecMrktInd']

Columns only in SBA 504:
['CDC_City', 'CDC_Name', 'CDC_State', 'CDC_Street', 'CDC_Zip', 'ThirdPartyDollars', 'ThirdPartyLender_City', 'ThirdPartyLender_Name', 'ThirdPartyLender_State']

Common columns:
['ApprovalDate', 'ApprovalFY', 'AsOfDate', 'BorrCity', 'BorrName', 'BorrState', 'BorrStreet', 'BorrZip', 'BusinessAge', 'BusinessType', 'ChargeOffDate', 'CollateralInd', 'CongressionalDistrict', 'FirstDisbursementDate', 'FranchiseCode', 'FranchiseName', 'GrossApproval', 'GrossChargeOffAmount', 'JobsSupported', 'LoanStatus', 'LocationID', 'NaicsCode', 'NaicsDescription', 'PaidInFullDate', 'ProcessingMethod', 'Program', 'ProjectCounty', 'ProjectState', 'SBADistrictOffice', 'TermInMonths']


## Schema Mapping

In [10]:
# Identify common and program-specific columns

schema_7a = program_schemas["7a"][0][1]
schema_504 = program_schemas["504"][0][1]

common_columns = sorted(schema_7a & schema_504)
only_7a = sorted(schema_7a - schema_504)
only_504 = sorted(schema_504 - schema_7a)

union_columns = sorted(schema_7a | schema_504)

print("Common columns:", len(common_columns))
print("7(a)-specific columns:", len(only_7a))
print("504-specific columns:", len(only_504))
print("Union schema:", len(union_columns))

Common columns: 30
7(a)-specific columns: 12
504-specific columns: 9
Union schema: 51


In [11]:
schema_mapping = pd.DataFrame({
    "column_name": union_columns
})

schema_mapping["availability"] = schema_mapping[
    "column_name"
].apply(
    lambda col:
        "Common" if col in common_columns
        else "7(a) only" if col in only_7a
        else "504 only"
)

display(schema_mapping)

,column_name,availability
0,ApprovalDate,Common
1,ApprovalFY,Common
2,AsOfDate,Common
3,BankCity,7(a) only
4,BankFDICNumber,7(a) only
5,BankNCUANumber,7(a) only
6,BankName,7(a) only
7,BankState,7(a) only
8,BankStreet,7(a) only
9,BankZip,7(a) only


In [12]:
for file in csv_files:

    df_sample = pd.read_csv(
        file,
        nrows=5,
        low_memory=False
    )

    print(f"\n===== {file.name} =====")
    print(df_sample.columns.tolist())
    display(df_sample.head(2))


===== FOIA_504_FY1991_FY2009_asof_260930.csv =====
['AsOfDate', 'Program', 'LocationID', 'BorrName', 'BorrStreet', 'BorrCity', 'BorrState', 'BorrZip', 'CDC_Name', 'CDC_Street', 'CDC_City', 'CDC_State', 'CDC_Zip', 'ThirdPartyLender_Name', 'ThirdPartyLender_City', 'ThirdPartyLender_State', 'ThirdPartyDollars', 'GrossApproval', 'ApprovalDate', 'ApprovalFY', 'FirstDisbursementDate', 'ProcessingMethod', 'TermInMonths', 'NaicsCode', 'NaicsDescription', 'FranchiseCode', 'FranchiseName', 'ProjectCounty', 'ProjectState', 'SBADistrictOffice', 'CongressionalDistrict', 'BusinessType', 'BusinessAge', 'LoanStatus', 'PaidInFullDate', 'ChargeOffDate', 'GrossChargeOffAmount', 'JobsSupported', 'CollateralInd']


,AsOfDate,Program,LocationID,BorrName,BorrStreet,BorrCity,BorrState,BorrZip,CDC_Name,CDC_Street,...,SBADistrictOffice,CongressionalDistrict,BusinessType,BusinessAge,LoanStatus,PaidInFullDate,ChargeOffDate,GrossChargeOffAmount,JobsSupported,CollateralInd
0,2026-09-30,504,188244,JACK BINGHAM PHOTOGRAPHY,SCRUTON POND ROAD,BARRINGTON,NH,3825,Granite State Economic Development Corporation,One Cate Street,...,NEW HAMPSHIRE DISTRICT OFFICE,1,INDIVIDUAL,Less than 4 years old but at least 3,PIF,2005-05-31,NaN,0.0,4.0,NaN
1,2026-09-30,504,188200,TKF Enterprises Inc,40TH ST. & DONALD PARKWAY,FAIRFIELD,AL,35208,Southern Development Council,200 Office Park Dr.,...,ALABAMA DISTRICT OFFICE,7,CORPORATION,Less than 4 years old but at least 3,CANCLD,NaN,NaN,0.0,15.0,NaN



===== FOIA_504_FY2010_Present_asof_260930.csv =====
['AsOfDate', 'Program', 'LocationID', 'BorrName', 'BorrStreet', 'BorrCity', 'BorrState', 'BorrZip', 'CDC_Name', 'CDC_Street', 'CDC_City', 'CDC_State', 'CDC_Zip', 'ThirdPartyLender_Name', 'ThirdPartyLender_City', 'ThirdPartyLender_State', 'ThirdPartyDollars', 'GrossApproval', 'ApprovalDate', 'ApprovalFY', 'FirstDisbursementDate', 'ProcessingMethod', 'TermInMonths', 'NaicsCode', 'NaicsDescription', 'FranchiseCode', 'FranchiseName', 'ProjectCounty', 'ProjectState', 'SBADistrictOffice', 'CongressionalDistrict', 'BusinessType', 'BusinessAge', 'LoanStatus', 'PaidInFullDate', 'ChargeOffDate', 'GrossChargeOffAmount', 'JobsSupported', 'CollateralInd']


,AsOfDate,Program,LocationID,BorrName,BorrStreet,BorrCity,BorrState,BorrZip,CDC_Name,CDC_Street,...,SBADistrictOffice,CongressionalDistrict,BusinessType,BusinessAge,LoanStatus,PaidInFullDate,ChargeOffDate,GrossChargeOffAmount,JobsSupported,CollateralInd
0,2026-09-30,504,188129,70 West Main Street Realty Corp,68-70 West Main Street.,Patchogue,NY,11772,Empire State Certified Development Corporation,19 British American Blvd. E.,...,NEW YORK DISTRICT OFFICE,1,CORPORATION,Less than 4 years old but at least 3,EXEMPT,NaN,NaN,0.0,5.0,Y
1,2026-09-30,504,188242,Martin Harper P.C.,3 E. Ramona Avenue.,Colorado springs,CO,80905,Pikes Peak Regional Development Corporation,322 S. Cascade Avenue,...,COLORADO DISTRICT OFFICE,5,CORPORATION,Less than 4 years old but at least 3,PIF,2020-11-30,NaN,0.0,1.0,Y



===== FOIA_7a_FY1991_FY1999_asof_260930.csv =====
['AsOfDate', 'Program', 'LocationID', 'BorrName', 'BorrStreet', 'BorrCity', 'BorrState', 'BorrZip', 'BankName', 'BankFDICNumber', 'BankNCUANumber', 'BankStreet', 'BankCity', 'BankState', 'BankZip', 'GrossApproval', 'SBAGuaranteedApproval', 'ApprovalDate', 'ApprovalFY', 'FirstDisbursementDate', 'ProcessingMethod', 'InitialInterestRate', 'FixedorVariableInterestInd', 'TermInMonths', 'NaicsCode', 'NaicsDescription', 'FranchiseCode', 'FranchiseName', 'ProjectCounty', 'ProjectState', 'SBADistrictOffice', 'CongressionalDistrict', 'BusinessType', 'BusinessAge', 'LoanStatus', 'PaidInFullDate', 'ChargeOffDate', 'GrossChargeOffAmount', 'RevolverStatus', 'JobsSupported', 'CollateralInd', 'SoldSecMrktInd']


,AsOfDate,Program,LocationID,BorrName,BorrStreet,BorrCity,BorrState,BorrZip,BankName,BankFDICNumber,...,BusinessType,BusinessAge,LoanStatus,PaidInFullDate,ChargeOffDate,GrossChargeOffAmount,RevolverStatus,JobsSupported,CollateralInd,SoldSecMrktInd
0,2026-09-30,7A,225134,"ROBINS INSURANCE AGENCY, INC.",30 BURTON HILLS BLVD.,NASHVILLE,TN,37215,Truist Bank,9846.0,...,CORPORATION,Less than 4 years old but at least 3,P I F,2005-07-31,NaN,0.0,N,0.0,NaN,NaN
1,2026-09-30,7A,5570,KESSLER SHELL SERVICE,16801 VENTURA BLVD.,ENCINO,CA,91316,General Bank,NaN,...,CORPORATION,Less than 4 years old but at least 3,P I F,2005-07-31,NaN,0.0,N,0.0,NaN,Y


... (2 outputs left out)


===== FOIA_7a_FY2010_FY2019_asof_260930.csv =====
['AsOfDate', 'Program', 'LocationID', 'BorrName', 'BorrStreet', 'BorrCity', 'BorrState', 'BorrZip', 'BankName', 'BankFDICNumber', 'BankNCUANumber', 'BankStreet', 'BankCity', 'BankState', 'BankZip', 'GrossApproval', 'SBAGuaranteedApproval', 'ApprovalDate', 'ApprovalFY', 'FirstDisbursementDate', 'ProcessingMethod', 'InitialInterestRate', 'FixedorVariableInterestInd', 'TermInMonths', 'NaicsCode', 'NaicsDescription', 'FranchiseCode', 'FranchiseName', 'ProjectCounty', 'ProjectState', 'SBADistrictOffice', 'CongressionalDistrict', 'BusinessType', 'BusinessAge', 'LoanStatus', 'PaidInFullDate', 'ChargeOffDate', 'GrossChargeOffAmount', 'RevolverStatus', 'JobsSupported', 'CollateralInd', 'SoldSecMrktInd']


,AsOfDate,Program,LocationID,BorrName,BorrStreet,BorrCity,BorrState,BorrZip,BankName,BankFDICNumber,...,BusinessType,BusinessAge,LoanStatus,PaidInFullDate,ChargeOffDate,GrossChargeOffAmount,RevolverStatus,JobsSupported,CollateralInd,SoldSecMrktInd
0,2026-09-30,7A,12096,"United Renal Associates of Georgia, P.C.","5669 Peachtree Dunwoody Road NE, Suite 240",Atlanta,GA,30342,Wells Fargo Bank National Association,3511.0,...,CORPORATION,"New, Less than 1 Year old",P I F,2010-11-30,NaN,0.0,Y,6.0,N,NaN
1,2026-09-30,7A,89255,Sylvia H. Caruso,"5 BROADWAY, SUITE 104",SAUGUS,MA,1906,TruNorth Bank,26484.0,...,INDIVIDUAL,"Existing, 5 or more years",P I F,2013-11-30,NaN,0.0,N,1.0,Y,NaN



===== FOIA_7a_FY2020_Present_asof_260930.csv =====
['AsOfDate', 'Program', 'LocationID', 'BorrName', 'BorrStreet', 'BorrCity', 'BorrState', 'BorrZip', 'BankName', 'BankFDICNumber', 'BankNCUANumber', 'BankStreet', 'BankCity', 'BankState', 'BankZip', 'GrossApproval', 'SBAGuaranteedApproval', 'ApprovalDate', 'ApprovalFY', 'FirstDisbursementDate', 'ProcessingMethod', 'InitialInterestRate', 'FixedorVariableInterestInd', 'TermInMonths', 'NaicsCode', 'NaicsDescription', 'FranchiseCode', 'FranchiseName', 'ProjectCounty', 'ProjectState', 'SBADistrictOffice', 'CongressionalDistrict', 'BusinessType', 'BusinessAge', 'LoanStatus', 'PaidInFullDate', 'ChargeOffDate', 'GrossChargeOffAmount', 'RevolverStatus', 'JobsSupported', 'CollateralInd', 'SoldSecMrktInd']


,AsOfDate,Program,LocationID,BorrName,BorrStreet,BorrCity,BorrState,BorrZip,BankName,BankFDICNumber,...,BusinessType,BusinessAge,LoanStatus,PaidInFullDate,ChargeOffDate,GrossChargeOffAmount,RevolverStatus,JobsSupported,CollateralInd,SoldSecMrktInd
0,2026-09-30,7A,26378,Rambler Concepts LLC,10124 Knoll Circle,LITTLETON,CO,80130,Sunflower Bank National Association,4767.0,...,CORPORATION,Change of Ownership,P I F,2021-08-31,NaN,0.0,N,2.0,Y,NaN
1,2026-09-30,7A,95003,Froyos & Vinos LLC,2864 B Highway 54,Peachtree City,GA,30269,Georgia's Own Credit Union,NaN,...,CORPORATION,"Startup, Loan Funds will Open Business",EXEMPT,NaN,NaN,0.0,N,10.0,Y,NaN


## Record Counting

In [13]:
record_counts = []

for file in csv_files:
    total_rows = 0

    for chunk in pd.read_csv(
        file,
        chunksize=100_000,
        low_memory=False
    ):
        total_rows += len(chunk)

    record_counts.append({
        "file_name": file.name,
        "program": "7(a)" if "7a_" in file.name.lower() else "504",
        "total_rows": total_rows
    })

df_record_counts = pd.DataFrame(record_counts)

display(df_record_counts)

,file_name,program,total_rows
0,FOIA_504_FY1991_FY2009_asof_260930.csv,504,111066
1,FOIA_504_FY2010_Present_asof_260930.csv,504,119599
2,FOIA_7a_FY1991_FY1999_asof_260930.csv,7(a),337033
3,FOIA_7a_FY2000_FY2009_asof_260930.csv,7(a),690333
4,FOIA_7a_FY2010_FY2019_asof_260930.csv,7(a),545751
5,FOIA_7a_FY2020_Present_asof_260930.csv,7(a),403644


In [14]:
program_summary = (
    df_record_counts
    .groupby("program")["total_rows"]
    .sum()
    .reset_index()
)

display(program_summary)

print("Total seluruh record:", df_record_counts["total_rows"].sum())

,program,total_rows
0,504,230665
1,7(a),1976761


Total seluruh record: 2207426


## COncenation Per-Program

In [15]:
STAGING_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_7A = STAGING_DIR / "sba_7a_combined.csv"
OUTPUT_504 = STAGING_DIR / "sba_504_combined.csv"

In [16]:
files_7a = sorted([
    file for file in csv_files
    if file.name.lower().startswith("foia_7a")
])

files_504 = sorted([
    file for file in csv_files
    if file.name.lower().startswith("foia_504")
])


def concatenate_csv_files(files, output_path):
    # Hapus output lama jika proses dijalankan ulang
    if output_path.exists():
        output_path.unlink()

    total_rows = 0
    write_header = True

    for file in files:
        print(f"Processing: {file.name}")

        for chunk in pd.read_csv(
            file,
            chunksize=100_000,
            dtype="string",
            keep_default_na=False,
            na_filter=False
        ):
            chunk.to_csv(
                output_path,
                mode="a",
                index=False,
                header=write_header,
                encoding="utf-8"
            )

            total_rows += len(chunk)
            write_header = False

    print(f"Completed: {output_path.name}")
    print(f"Total rows: {total_rows:,}")

    return total_rows


# Concatenate 7(a)
rows_7a = concatenate_csv_files(
    files_7a,
    OUTPUT_7A
)

# Concatenate 504
rows_504 = concatenate_csv_files(
    files_504,
    OUTPUT_504
)

Processing: FOIA_7a_FY1991_FY1999_asof_260930.csv
Processing: FOIA_7a_FY2000_FY2009_asof_260930.csv
Processing: FOIA_7a_FY2010_FY2019_asof_260930.csv
Processing: FOIA_7a_FY2020_Present_asof_260930.csv
Completed: sba_7a_combined.csv
Total rows: 1,976,761
Processing: FOIA_504_FY1991_FY2009_asof_260930.csv
Processing: FOIA_504_FY2010_Present_asof_260930.csv
Completed: sba_504_combined.csv
Total rows: 230,665


In [17]:
expected_7a = df_record_counts.loc[
    df_record_counts["program"] == "7(a)",
    "total_rows"
].sum()

expected_504 = df_record_counts.loc[
    df_record_counts["program"] == "504",
    "total_rows"
].sum()

validation = pd.DataFrame({
    "program": ["7(a)", "504"],
    "expected_rows": [expected_7a, expected_504],
    "actual_rows": [rows_7a, rows_504]
})

validation["match"] = (
    validation["expected_rows"] == validation["actual_rows"]
)

display(validation)

,program,expected_rows,actual_rows,match
0,7(a),1976761,1976761,True
1,504,230665,230665,True


## Final COmbined Dataset

In [18]:
# Urutan kolom: kolom 7(a) terlebih dahulu, lalu kolom khusus 504
all_source_files = files_7a + files_504

union_columns = list(dict.fromkeys(
    column
    for file in all_source_files
    for column in pd.read_csv(file, nrows=0).columns
))

print("Jumlah union columns:", len(union_columns))
print(union_columns)

Jumlah union columns: 51
['AsOfDate', 'Program', 'LocationID', 'BorrName', 'BorrStreet', 'BorrCity', 'BorrState', 'BorrZip', 'BankName', 'BankFDICNumber', 'BankNCUANumber', 'BankStreet', 'BankCity', 'BankState', 'BankZip', 'GrossApproval', 'SBAGuaranteedApproval', 'ApprovalDate', 'ApprovalFY', 'FirstDisbursementDate', 'ProcessingMethod', 'InitialInterestRate', 'FixedorVariableInterestInd', 'TermInMonths', 'NaicsCode', 'NaicsDescription', 'FranchiseCode', 'FranchiseName', 'ProjectCounty', 'ProjectState', 'SBADistrictOffice', 'CongressionalDistrict', 'BusinessType', 'BusinessAge', 'LoanStatus', 'PaidInFullDate', 'ChargeOffDate', 'GrossChargeOffAmount', 'RevolverStatus', 'JobsSupported', 'CollateralInd', 'SoldSecMrktInd', 'CDC_Name', 'CDC_Street', 'CDC_City', 'CDC_State', 'CDC_Zip', 'ThirdPartyLender_Name', 'ThirdPartyLender_City', 'ThirdPartyLender_State', 'ThirdPartyDollars']


In [19]:
OUTPUT_COMBINED = STAGING_DIR / "sba_full_combined.csv"

if OUTPUT_COMBINED.exists():
    OUTPUT_COMBINED.unlink()

total_rows = 0
write_header = True

for file in all_source_files:
    print(f"Processing: {file.name}")

    for chunk in pd.read_csv(
        file,
        chunksize=100_000,
        dtype="string",
        keep_default_na=False,
        na_filter=False
    ):
        # Selaraskan schema berdasarkan nama kolom
        chunk = chunk.reindex(columns=union_columns)

        # Tambahkan metadata sumber
        chunk["SourceFile"] = file.name

        # Tulis ke output
        chunk.to_csv(
            OUTPUT_COMBINED,
            mode="a",
            index=False,
            header=write_header,
            encoding="utf-8"
        )

        total_rows += len(chunk)
        write_header = False

    print(f"Completed: {file.name}")

print(f"\nFinal dataset created: {OUTPUT_COMBINED}")
print(f"Total rows written: {total_rows:,}")
print(f"Total columns: {len(union_columns) + 1}")

Processing: FOIA_7a_FY1991_FY1999_asof_260930.csv
Completed: FOIA_7a_FY1991_FY1999_asof_260930.csv
Processing: FOIA_7a_FY2000_FY2009_asof_260930.csv
Completed: FOIA_7a_FY2000_FY2009_asof_260930.csv
Processing: FOIA_7a_FY2010_FY2019_asof_260930.csv
Completed: FOIA_7a_FY2010_FY2019_asof_260930.csv
Processing: FOIA_7a_FY2020_Present_asof_260930.csv
Completed: FOIA_7a_FY2020_Present_asof_260930.csv
Processing: FOIA_504_FY1991_FY2009_asof_260930.csv
Completed: FOIA_504_FY1991_FY2009_asof_260930.csv
Processing: FOIA_504_FY2010_Present_asof_260930.csv
Completed: FOIA_504_FY2010_Present_asof_260930.csv

Final dataset created: d:\Project\credit-risk-monitoring\assets\data\staging\sba\sba_full_combined.csv
Total rows written: 2,207,426
Total columns: 52


In [20]:
# Periksa header
final_columns = pd.read_csv(
    OUTPUT_COMBINED,
    nrows=0
).columns.tolist()

print("Jumlah kolom:", len(final_columns))
print("SourceFile tersedia:", "SourceFile" in final_columns)

# Periksa jumlah record dan distribusi sumber
source_counts = {}

for chunk in pd.read_csv(
    OUTPUT_COMBINED,
    chunksize=100_000,
    usecols=["SourceFile"],
    dtype="string"
):
    counts = chunk["SourceFile"].value_counts()

    for source, count in counts.items():
        source_counts[source] = source_counts.get(source, 0) + count

df_source_validation = pd.DataFrame([
    {
        "file_name": source,
        "expected_rows": int(
            df_record_counts.loc[
                df_record_counts["file_name"] == source,
                "total_rows"
            ].iloc[0]
        ),
        "actual_rows": count
    }
    for source, count in source_counts.items()
])

df_source_validation["match"] = (
    df_source_validation["expected_rows"]
    == df_source_validation["actual_rows"]
)

display(df_source_validation)

print("Total actual rows:", sum(source_counts.values()))
print("Total columns:", len(final_columns))
print("All source counts match:", df_source_validation["match"].all())

Jumlah kolom: 52
SourceFile tersedia: True


,file_name,expected_rows,actual_rows,match
0,FOIA_7a_FY1991_FY1999_asof_260930.csv,337033,337033,True
1,FOIA_7a_FY2000_FY2009_asof_260930.csv,690333,690333,True
2,FOIA_7a_FY2010_FY2019_asof_260930.csv,545751,545751,True
3,FOIA_7a_FY2020_Present_asof_260930.csv,403644,403644,True
4,FOIA_504_FY1991_FY2009_asof_260930.csv,111066,111066,True
5,FOIA_504_FY2010_Present_asof_260930.csv,119599,119599,True


Total actual rows: 2207426
Total columns: 52
All source counts match: True
